In [16]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import platform
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import psutil
import tensorflow as tf
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.preprocessing import MinMaxScaler, StandardScaler

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TARGET_COL = "attack_cat"
N_REPEATS = 3


In [17]:
# Carga de dataset y preprocesado, igual que en los cuadernos de ataques para UNSW-NB15
path_train = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv"
path_test = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv"

df_train = pl.read_csv(path_train)
df_test = pl.read_csv(path_test)

y_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

y_test = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

x_train = df_train.drop(TARGET_COL)
x_test = df_test.drop(TARGET_COL)

X_train_raw = x_train.to_numpy().astype(np.float32)
X_test_raw = x_test.to_numpy().astype(np.float32)
X_full_raw = np.vstack([X_train_raw, X_test_raw]).astype(np.float32)

y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()
y_full_np = np.concatenate([y_train_np, y_test_np])

mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_train_raw).astype(np.float32)
X_test_scaled_mlp = mlp_scaler.transform(X_test_raw).astype(np.float32)
X_full_scaled_mlp = mlp_scaler.transform(X_full_raw).astype(np.float32)

cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_train_raw).astype(np.float32)
X_test_scaled_cnn = cnn_scaler.transform(X_test_raw).astype(np.float32)
X_full_scaled_cnn = cnn_scaler.transform(X_full_raw).astype(np.float32)

X_train_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)
X_full_cnn = X_full_scaled_cnn.reshape(X_full_scaled_cnn.shape[0], X_full_scaled_cnn.shape[1], 1)


In [18]:
MODEL_INPUTS = {
    "test": {
        "rf": X_test_raw,
        "xgb": X_test_raw,
        "lgbm": X_test_raw,
        "catboost": X_test_raw,
        "svm": X_test_scaled_mlp,
        "mlp": X_test_scaled_mlp,
        "cnn": X_test_cnn,
    },
    "full": {
        "rf": X_full_raw,
        "xgb": X_full_raw,
        "lgbm": X_full_raw,
        "catboost": X_full_raw,
        "svm": X_full_scaled_mlp,
        "mlp": X_full_scaled_mlp,
        "cnn": X_full_cnn,
    },
}


In [19]:
# Tamaño de los datasets

print(len(X_full_raw))

257673


In [20]:
# RF

rf_path = "../model/unsw-nb15/rf_unsw.joblib"

rf_model = joblib.load(rf_path)
print("Modelo Cargado")

# Warm-up
_ = rf_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_rf = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_rf.append(t1 - t0)

tiempo_total_rf = float(np.mean(tiempos_rf))

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_rf]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_rf:.4f} s")

Modelo Cargado
Tiempos medidos: [0.1817, 0.177, 0.1633]
Tiempo medio de inferencia sobre todo el dataset: 0.1740 s


In [21]:
# XGBOOST

xgb_path = "../model/unsw-nb15/xgb_unsw.joblib"

xgb_model = joblib.load(xgb_path)
print("Modelo Cargado")

# Warm-up
_ = xgb_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_xgb = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = xgb_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_xgb.append(t1 - t0)

tiempo_total_xgb = float(np.mean(tiempos_xgb))

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_xgb]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_xgb:.4f} s")

Modelo Cargado


/usr/lib/python3.12/pickle.py:1710: UserWarning: [23:17:07] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1710: UserWarning: [23:17:07] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1710: UserWarning: [23:17:07] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


Tiempos medidos: [0.2138, 0.2041, 0.204]
Tiempo medio de inferencia sobre todo el dataset: 0.2073 s


In [22]:
# LIGHTGBM

lgbm_path = "../model/unsw-nb15/lgbm_unsw.joblib"

lgbm_model = joblib.load(lgbm_path)
print("Modelo Cargado")

# Warm-up
_ = lgbm_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_lgbm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = lgbm_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_lgbm.append(t1 - t0)

tiempo_total_lgbm = float(np.mean(tiempos_lgbm))

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_lgbm]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_lgbm:.4f} s")

Modelo Cargado


/home/placi/Escritorio/TFG/PLACI_TFG/CODIGO/tfg_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placi/Escritorio/TFG/PLACI_TFG/CODIGO/tfg_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placi/Escritorio/TFG/PLACI_TFG/CODIGO/tfg_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Tiempos medidos: [0.2006, 0.1844, 0.1904]
Tiempo medio de inferencia sobre todo el dataset: 0.1918 s


/home/placi/Escritorio/TFG/PLACI_TFG/CODIGO/tfg_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [23]:
# CATBOOST

catboost_path = "../model/unsw-nb15/catboost_unsw.joblib"

catboost_model = joblib.load(catboost_path)
print("Modelo Cargado")

# Warm-up
_ = catboost_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_catboost = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = catboost_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_catboost.append(t1 - t0)

tiempo_total_catboost = float(np.mean(tiempos_catboost))

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_catboost]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_catboost:.4f} s")



Modelo Cargado
Tiempos medidos: [0.1355, 0.1186, 0.1158]
Tiempo medio de inferencia sobre todo el dataset: 0.1233 s


In [24]:
# SVM

svm_path = "../model/unsw-nb15/svm_unsw.joblib"

svm_model = joblib.load(svm_path)
print("Modelo Cargado")

# Warm-up
_ = svm_model.predict(X_full_scaled_mlp[:min(1000, len(X_full_scaled_mlp))])

# Medida Latencia
tiempos_svm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = svm_model.predict(X_full_scaled_mlp)
    t1 = time.perf_counter()
    tiempos_svm.append(t1 - t0)

tiempo_total_svm = float(np.mean(tiempos_svm))

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_svm]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_svm:.4f} s")

Modelo Cargado
Tiempos medidos: [0.0161, 0.012, 0.0219]
Tiempo medio de inferencia sobre todo el dataset: 0.0167 s


In [25]:
# MLP

mlp_path = "../model/unsw-nb15/mlp_unsw.joblib"

mlp_model = joblib.load(mlp_path)
print("Modelo Cargado")

# Warm-up
_ = mlp_model.predict(X_full_scaled_mlp[:min(1000, len(X_full_scaled_mlp))], batch_size=4096, verbose=0)

# Medida Latencia
tiempos_mlp = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = mlp_model.predict(X_full_scaled_mlp, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_mlp.append(t1 - t0)

tiempo_total_mlp = float(np.mean(tiempos_mlp))

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_mlp]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_mlp:.4f} s")

E0000 00:00:1779311828.833166    4432 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Modelo Cargado


I0000 00:00:1779311829.167385    5985 service.cc:153] XLA service 0x7e3f4c031ec0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779311829.167403    5985 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1779311829.186066    5985 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779311829.322136    5985 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Tiempos medidos: [0.3647, 0.1263, 0.1177]
Tiempo medio de inferencia sobre todo el dataset: 0.2029 s


In [26]:
# CNN

cnn_path = "../model/unsw-nb15/cnn_unsw.joblib"

cnn_model = joblib.load(cnn_path)
print("Modelo Cargado")

# Warm-up
_ = cnn_model.predict(X_full_cnn[:min(1000, len(X_full_cnn))], batch_size=4096, verbose=0)

# Medida Latencia
tiempos_cnn = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = cnn_model.predict(X_full_cnn, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_cnn.append(t1 - t0)

tiempo_total_cnn = float(np.mean(tiempos_cnn))

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_cnn]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_cnn:.4f} s")


Modelo Cargado
Tiempos medidos: [1.0068, 0.7259, 0.8867]
Tiempo medio de inferencia sobre todo el dataset: 0.8731 s
